In [1]:
from argparse import ArgumentParser
import xarray as xr
from matplotlib import pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys

sys.path.insert(
    1, "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/evaluation"
)
import plot_utils as plot
import numpy as np


def create_folder_if_not_exists(folder):
    from pathlib import Path

    Path(folder).mkdir(parents=True, exist_ok=True)


def get_temporal_index_in_dataset(timestamp, ds):
    """
    Returns the integer index of a given timestamp in an xarray Dataset
    that may use non-Gregorian calendars (e.g., 360_day).

    timestamp: cftime.Datetime object or a string parseable by cftime
    ds: xarray.Dataset or DataArray with a 'time' coordinate
    """
    import cftime

    # Iterate over time coordinates to find match
    for i, t in enumerate(ds.time.values):
        if t == timestamp:
            return i

    # If not found, raise error
    raise ValueError(f"Timestamp {timestamp} not found in dataset time axis.")


def get_true_data_matching_forecast_timestamps(
    ds_truth, ds_forecast, prediction_steps
):
    """
    prediction_steps should be 0 based -> 0 means using the first prediction of the model for that timestamp
    """
    first_timestamp_forecast = ds_forecast.time[0].values
    last_timestamp_forecast = ds_forecast.time[-1].values
    idx_truth_first = (
        get_temporal_index_in_dataset(first_timestamp_forecast, ds_truth)
        + prediction_steps
        + 1
    )
    idx_truth_last = (
        get_temporal_index_in_dataset(last_timestamp_forecast, ds_truth)
        + prediction_steps
        + 1
    )
    new_forecast = ds_forecast.copy().isel(
        prediction_timedelta=prediction_steps
    )
    new_truth = ds_truth.isel(
        time=slice(idx_truth_first, idx_truth_last + 1)
    ).isel(time=slice(0, None, 2))
    new_forecast = new_forecast.sortby("time")
    _, index = np.unique(new_forecast["time"], return_index=True)
    new_forecast = new_forecast.isel(time=np.sort(index))

    for t in range(len(new_forecast.time.values) - 1):
        if (
            not str(
                new_forecast.time.values[t + 1] - new_forecast.time.values[t]
            )
            == "12:00:00"
        ):
            print(
                f"Error with {new_forecast.time.values[t]} and {new_forecast.time.values[t+1]}"
            )
    new_forecast = new_forecast.assign_coords({"time": new_truth["time"]})
    return new_truth, new_forecast


def get_stacked_true_data_matching_forecast_timestamps(ds_truth, ds_forecast):
    """
    prediction_steps should be 0 based -> 0 means using the first prediction of the model for that timestamp
    """
    first_timestamp_forecast = ds_forecast.time[0].values
    last_timestamp_forecast = ds_forecast.time[-1].values
    idx_truth_first = get_temporal_index_in_dataset(
        first_timestamp_forecast, ds_truth
    )
    idx_truth_last = get_temporal_index_in_dataset(
        last_timestamp_forecast, ds_truth
    )
    new_forecast = ds_forecast.copy()
    new_truth = ds_truth.isel(
        time=slice(idx_truth_first, idx_truth_last + 1)
    ).isel(time=slice(0, None, 2))
    new_forecast = new_forecast.sortby("time")
    _, index = np.unique(new_forecast["time"], return_index=True)
    new_forecast = new_forecast.isel(time=np.sort(index))
    new_forecast = new_forecast.chunk({"latitude": -1, "longitude": -1})
    new_truth = new_truth.chunk({"latitude": -1, "longitude": -1})
    # for t in range(len(new_forecast.time.values)-1):
    #     if not str(new_forecast.time.values[t+1] - new_forecast.time.values[t]) == '12:00:00':
    #         print(f'Error with {new_forecast.time.values[t]} and {new_forecast.time.values[t+1]}')
    # new_forecast = new_forecast.assign_coords({'time': new_truth['time']})
    return new_truth, new_forecast


def build_truth_with_leadtime(truth_ds, max_lead_steps=40):
    """
    truth_ds: Dataset with dimension (time, lat, lon, level)
    Creates a dataset with dimension (time, prediction_timedelta, lat, lon, level)

    lead step k uses truth_ds shifted by k steps forward in time.
    """

    lead_times = np.arange(1, max_lead_steps + 1)

    # Build stacked truth array
    truth_list = []
    for k in lead_times:
        truth_shifted = truth_ds.shift(
            time=-k
        )  # shift backwards so truth[t,k] = truth[t+k]
        truth_list.append(truth_shifted)

    # Combine along new dimension
    truth_with_lead = xr.concat(truth_list, dim="prediction_timedelta")

    # Assign actual timedelta
    truth_with_lead = truth_with_lead.assign_coords(
        prediction_timedelta=xr.DataArray(
            (lead_times * np.timedelta64(6, "h")).astype("timedelta64[ns]"),
            dims=("prediction_timedelta",),
        )
    )

    # Drop times near the end that became NaN
    truth_with_lead = truth_with_lead.dropna(dim="time", how="any")

    return truth_with_lead


def compute_r2_vectorized(model_idx=0, past_int=0):

    past = False if past_int == 0 else True
    dataset_slice = (
        plot.DATASET_SLICE.TEST_HIST if past else plot.DATASET_SLICE.TEST_FUTURE
    )
    model_resolution = [
        "z64",
        "z128",
        "z256",
        "z64 hierarchical",
        "persistence",
    ]

    current_model_resolution = model_resolution[model_idx]
    print(
        f'Calculating r2 scores for model {current_model_resolution} in {"past" if past else "future"}'
    )

    selected_model_past = plot.PAST_MODELS_UKESM[model_idx]
    selected_model_future = plot.FUTURE_MODELS_UKESM[model_idx]
    print(f'Evaluating model: {selected_model_past["model_name"]}')

    base_save_folder_past = selected_model_past["plot_folder"]
    base_save_folder_future = selected_model_future["plot_folder"]

    ukesm_past_forecasts = xr.open_zarr(
        selected_model_past["forecasts"], decode_timedelta=True
    ).astype(np.float32)
    ukesm_future_forecasts = xr.open_zarr(
        selected_model_future["forecasts"], decode_timedelta=True
    ).astype(np.float32)

    ukesm_past_truth = xr.open_zarr(selected_model_past["ground_truth"]).astype(
        np.float32
    )
    ukesm_future_truth = xr.open_zarr(
        selected_model_future["ground_truth"]
    ).astype(np.float32)

    create_folder_if_not_exists(base_save_folder_past)
    create_folder_if_not_exists(base_save_folder_future)

    print("Stacking truth data...")
    truth_stacked = build_truth_with_leadtime(
        ukesm_past_truth if past else ukesm_future_truth
    )
    print("Selecting matching forecasts and truth...")
    truth_ds, forecast_ds = get_stacked_true_data_matching_forecast_timestamps(
        truth_stacked, ukesm_past_forecasts if past else ukesm_future_forecasts
    )
    lat = truth_ds["latitude"]
    weights = np.cos(np.deg2rad(lat))
    weights = xr.DataArray(weights, coords={"latitude": lat}, dims=["latitude"])

    # normalize weights so they sum to 1 for stability
    weights = weights / weights.sum()
    all_r2 = {}

    for key in range(len(plot.KEY_VARIABLES_UKESM)):
        display_name, unit, variable, plevel = plot.KEY_VARIABLES_UKESM[key]
        print(f"Current var: {display_name}")
        # --- Select variable ---
        truth = truth_ds[variable]
        forecast = forecast_ds[variable]

        # --- Level selection ---
        if plevel is not None:
            truth = truth.sel(level=plevel)
            forecast = forecast.sel(level=plevel)

            if plevel > 500:
                truth = plot.get_masked_ukesm_data(
                    truth, dataset_slice=dataset_slice, plevel=plevel
                )
                forecast = plot.get_masked_ukesm_data(
                    forecast, dataset_slice=dataset_slice, plevel=plevel
                )

        # --- Broadcast weights ---
        # weights(lat) → (lat, lon)
        W2d = weights.broadcast_like(truth.isel(time=0, prediction_timedelta=0))

        # W2d → (time, lead, lat, lon)
        W = W2d.broadcast_like(truth)

        # --- Weighted mean for each lead time ---
        # dims: ('time','lat','lon')
        truth_mean = (truth * W).sum(dim=("latitude", "longitude")) / W.sum(
            dim=("latitude", "longitude")
        )

        # truth_mean shape: (time, prediction_timedelta)

        # --- SSE & SST ---
        # subtract truth_mean along time dimension
        diff_truth = truth - truth_mean

        # sum over time, lat, lon
        sse = ((truth - forecast) ** 2 * W).sum(
            dim=("time", "latitude", "longitude")
        )
        sst = ((diff_truth) ** 2 * W).sum(dim=("time", "latitude", "longitude"))

        r2 = 1 - sse / sst

        # Convert xarray DataArray → numpy vector
        r2_values = r2.values  # shape (40,)
        all_r2[variable] = r2_values

        import pandas as pd

        df_r2 = pd.DataFrame(all_r2).T  # variables = rows, steps = columns
        df_r2.columns = [f"step_{i+1}" for i in range(40)]

        df_r2.to_csv(
            "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/values/"
            + f"r2_all_variables_{current_model_resolution}_{'past' if past else 'future'}_new.csv")



In [7]:
import pandas as pd
model_idx=2
model_resolution = [
    "z64",
    "z128",
    "z256",
    "z64 hierarchical",
    "persistence",
]

df_r2 = pd.read_csv('/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/values/' + f"rmse_all_variables_z256_past.csv", index_col=0)
df_r2_2 = pd.read_csv('/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/values/' + f"rmse_all_variables_z256_past7.csv", index_col=0)
for key in range(7,len(plot.KEY_VARIABLES_UKESM)): # Specific humidity
    display_name, unit, variable, plevel = plot.KEY_VARIABLES_UKESM[key]
    df_r2.loc[variable] = df_r2_2.loc[variable]
print(df_r2)


                             step_1      step_2      step_3      step_4  \
geopotential_500         172.434338  240.042744  309.334651  371.622492   
temperature               13.247535   18.689451   21.572770   23.484945   
specific_humidity          0.005457    0.008271    0.010091    0.011476   
surface_temperature       29.624684   39.901131   44.467631   46.948320   
u_component_of_wind       20.550579   28.207017   32.444905   35.640851   
v_component_of_wind       20.940182   28.590381   32.661073   35.803614   
total_precipitation_6hr   30.387602   42.436076   48.046257   52.440011   
relative_vorticity         0.000298    0.000411    0.000476    0.000523   

                             step_5      step_6      step_7      step_8  \
geopotential_500         438.925245  508.119908  578.141474  652.061450   
temperature               25.287589   26.759109   28.132027   29.432712   
specific_humidity          0.012713    0.013828    0.014859    0.015818   
surface_temperature     

In [8]:
df_r2.to_csv('/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/values/' + f"rmse_all_variables_z256_past_full.csv")

In [ ]:
def compute_rmse_vectorized(model_idx=0, past_int=0):

    past = False if past_int == 0 else True
    dataset_slice = (
        plot.DATASET_SLICE.TEST_HIST if past else plot.DATASET_SLICE.TEST_FUTURE
    )

    model_resolution = [
        "z64",
        "z128",
        "z256",
        "z64 hierarchical",
        "persistence",
    ]

    current_model_resolution = model_resolution[model_idx]
    print(
        f'Calculating RMSE for model {current_model_resolution} in {"past" if past else "future"}'
    )

    selected_model_past = plot.PAST_MODELS_UKESM[model_idx]
    selected_model_future = plot.FUTURE_MODELS_UKESM[model_idx]
    print(f'Evaluating model: {selected_model_past["model_name"]}')

    base_save_folder_past = selected_model_past["plot_folder"]
    base_save_folder_future = selected_model_future["plot_folder"]

    ukesm_past_forecasts = xr.open_zarr(
        selected_model_past["forecasts"], decode_timedelta=True
    ).astype(np.float32)
    ukesm_future_forecasts = xr.open_zarr(
        selected_model_future["forecasts"], decode_timedelta=True
    ).astype(np.float32)

    ukesm_past_truth = xr.open_zarr(selected_model_past["ground_truth"]).astype(
        np.float32
    )
    ukesm_future_truth = xr.open_zarr(
        selected_model_future["ground_truth"]
    ).astype(np.float32)

    create_folder_if_not_exists(base_save_folder_past)
    create_folder_if_not_exists(base_save_folder_future)

    print("Stacking truth...")
    truth_stacked = build_truth_with_leadtime(
        ukesm_past_truth if past else ukesm_future_truth
    )

    print("Selecting matching truth / forecasts...")
    truth_ds, forecast_ds = get_stacked_true_data_matching_forecast_timestamps(
        truth_stacked, ukesm_past_forecasts if past else ukesm_future_forecasts
    )

    print("Preparing weights...")
    lat = truth_ds["latitude"]
    weights = np.cos(np.deg2rad(lat))
    weights = xr.DataArray(weights, coords={"latitude": lat}, dims=["latitude"])
    weights = weights / weights.sum()

    all_rmse = {}
    rmse_maps = {}

    for key in range(len(plot.KEY_VARIABLES_UKESM)):

        display_name, unit, variable, plevel = plot.KEY_VARIABLES_UKESM[key]
        print(f"Current var (RMSE): {display_name}")

        # Select variable
        truth = truth_ds[variable]
        forecast = forecast_ds[variable]

        # Level selection
        if plevel is not None:
            truth = truth.sel(level=plevel)
            forecast = forecast.sel(level=plevel)

            if plevel > 500:
                truth = plot.get_masked_ukesm_data(
                    truth, dataset_slice=dataset_slice, plevel=plevel
                )
                forecast = plot.get_masked_ukesm_data(
                    forecast, dataset_slice=dataset_slice, plevel=plevel
                )

        # Broadcast weights
        W2d = weights.broadcast_like(truth.isel(time=0, prediction_timedelta=0))
        W = W2d.broadcast_like(truth)

        # Weighted MSE
        mse = ((forecast - truth) ** 2 * W).sum(
            dim=("time", "latitude", "longitude")
        ) / W2d.sum(dim=("latitude", "longitude"))

        rmse = np.sqrt(mse)

        mse_map = ((forecast - truth) ** 2).mean(dim="time")
        rmse_map = np.sqrt(mse_map)  # (prediction_timedelta, lat, lon)

        if plevel:
            rmse_maps[variable] = rmse_map.drop_vars("level")
        else:
            rmse_maps[variable] = rmse_map

        print(f"RMSE map for {variable}: {rmse_maps[variable].shape}")

        # Convert to numpy
        rmse_values = rmse.values  # shape (40,)
        all_rmse[variable] = rmse_values

        print(all_rmse[variable])
        # Save
        import pandas as pd

        df_rmse = pd.DataFrame(all_rmse).T
        df_rmse.columns = [f"step_{i+1}" for i in range(40)]

        save_path = (
            "/home/hk-project-pai00005/xo8179/neural_lam_fork/"
            "neural-lam/eval_ukesm/values/"
            f"rmse_all_variables_{current_model_resolution}_{'past' if past else 'future'}.csv"
        )

        df_rmse.to_csv(save_path)
        print("Saved:", save_path)

        ds_rmse_maps = xr.Dataset({var: rmse_maps[var] for var in rmse_maps})

        map_save = (
            "/home/hk-project-pai00005/xo8179/neural_lam_fork/"
            "neural-lam/eval_ukesm/values/"
            f"rmse_maps_{current_model_resolution}_{'past' if past else 'future'}.zarr"
        )
        ds_rmse_maps.to_zarr(map_save, mode="w")
        print("Saved RMSE maps:", map_save)

In [ ]:
compute_rmse_vectorized(0,0)


Calculating RMSE for model z64 in future
Evaluating model: UKESM z64 Multilevel Graph
Stacking truth...
Selecting matching truth / forecasts...
Preparing weights...
Loading mask...
Current var (RMSE): Temperature at 850hPa


/software/all/jupyter/ai/2025-05-23/lib/python3.11/site-packages/dask/array/core.py:4998: PerformanceWarning: Increasing number of chunks by factor of 20
  result = blockwise(
/software/all/jupyter/ai/2025-05-23/lib/python3.11/site-packages/dask/array/core.py:4998: PerformanceWarning: Increasing number of chunks by factor of 20
  result = blockwise(


<xarray.DataArray 'temperature' (prediction_timedelta: 40, longitude: 128,
                                 latitude: 64)> Size: 1MB
dask.array<sqrt, shape=(40, 128, 64), dtype=float32, chunksize=(1, 128, 64), chunktype=numpy.ndarray>
Coordinates:
  * latitude              (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
    level                 float32 4B 850.0
  * longitude             (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 320B 06:00:0...
RMSE map for temperature: (40, 128, 64)
